# 1. Analizando qualidade dos dados disponíveis

In [21]:
from pathlib import Path
import yaml
import pandas as pd
import requests
import numpy as np


PASTA_PROJETO = Path("raw_info")
PASTA_CHUNKS = PASTA_PROJETO / "chunks"
ARQUIVO_COMPLETO = PASTA_PROJETO / "highbond_api_completo.md"

arquivos_chunks = sorted(PASTA_CHUNKS.glob("*.md"))

print(f"Arquivo completo encontrado: {ARQUIVO_COMPLETO.exists()}")
print(f"Quantidade de arquivos em chunks: {len(arquivos_chunks)}")

Arquivo completo encontrado: True
Quantidade de arquivos em chunks: 370


In [22]:
def carregar_markdown(caminho):
    texto = caminho.read_text(encoding="utf-8")

    if not texto.startswith("---"):
        raise ValueError(f"Front matter não encontrado: {caminho.name}")

    partes = texto.split("---", 2)

    if len(partes) < 3:
        raise ValueError(f"Front matter inválido: {caminho.name}")

    metadados = yaml.safe_load(partes[1]) or {}
    conteudo = partes[2].strip()

    return {
        "arquivo": caminho.name,
        "id": metadados.get("id"),
        "title": metadados.get("title"),
        "tags": metadados.get("tags", []),
        "source": metadados.get("source"),
        "conteudo": conteudo,
        "caracteres": len(conteudo),
        "palavras": len(conteudo.split()),
        "tokens_estimados": round(len(conteudo) / 4),
    }


In [23]:
documentos = []

for caminho in arquivos_chunks:
    try:
        documentos.append(carregar_markdown(caminho))
    except Exception as erro:
        print(f"Erro em {caminho.name}: {erro}")

df_documentos = pd.DataFrame(documentos)

print(f"Média de tokens estimados por chunk: {df_documentos['tokens_estimados'].mean():.2f}")
df_documentos.head()

Erro em index.md: Front matter não encontrado: index.md
Média de tokens estimados por chunk: 1359.64


,arquivo,id,title,tags,source,conteudo,caracteres,palavras,tokens_estimados
0,endpoint-delete-orgs-org-id-actions-id-deletea...,endpoint-delete-orgs-org-id-actions-id-deletea...,Actions: Delete an action,"[endpoint, actions]",openapi.json (HighBond API v1.1.0),## Actions\n\nAn action is a specific follow-u...,2353,326,588
1,endpoint-delete-orgs-org-id-analyses-id-delete...,endpoint-delete-orgs-org-id-analyses-id-delete...,Analyses: Delete an analysis,"[endpoint, analyses]",openapi.json (HighBond API v1.1.0),## Analyses\n\nAn analysis is a second-level c...,1670,222,418
2,endpoint-delete-orgs-org-id-collaborator-group...,endpoint-delete-orgs-org-id-collaborator-group...,Collaborators: Delete a collaborator group,"[endpoint, collaborators]",openapi.json (HighBond API v1.1.0),## Collaborators\n\nA collaborator is a repres...,2157,295,539
3,endpoint-delete-orgs-org-id-collaborators-enco...,endpoint-delete-orgs-org-id-collaborators-enco...,Collaborators: Delete a collaborator,"[endpoint, collaborators]",openapi.json (HighBond API v1.1.0),## Collaborators\n\nA collaborator is a repres...,2117,279,529
4,endpoint-delete-orgs-org-id-collections-id-del...,endpoint-delete-orgs-org-id-collections-id-del...,Collections: Delete a collection,"[endpoint, collections]",openapi.json (HighBond API v1.1.0),## Collections\n\nA collection is a container ...,1716,225,429


In [24]:
print(f"Documentos carregados: {len(df_documentos)}")
print(f"IDs ausentes: {df_documentos['id'].isna().sum()}")
print(f"Títulos ausentes: {df_documentos['title'].isna().sum()}")
print(f"IDs duplicados: {df_documentos['id'].duplicated().sum()}")
print(f"Conteúdos vazios: {(df_documentos['caracteres'] == 0).sum()}")

Documentos carregados: 369
IDs ausentes: 0
Títulos ausentes: 0
IDs duplicados: 0
Conteúdos vazios: 0


In [25]:
df_documentos["tokens_estimados"].describe()

count      369.000000
mean      1359.642276
std       1418.281846
min        216.000000
25%        648.000000
50%        990.000000
75%       1546.000000
max      11836.000000
Name: tokens_estimados, dtype: float64

In [26]:
df_documentos[
    ["arquivo", "id", "title", "tokens_estimados"]
].sort_values(
    "tokens_estimados",
    ascending=False
)

,arquivo,id,title,tokens_estimados
301,endpoint-post-orgs-org-id-project-types-create...,endpoint-post-orgs-org-id-project-types-create...,Project types: Create or copy a project type,11836
245,endpoint-patch-orgs-org-id-project-types-id-up...,endpoint-patch-orgs-org-id-project-types-id-up...,Project types: Update a project type,11430
125,endpoint-get-orgs-org-id-project-types-getproj...,endpoint-get-orgs-org-id-project-types-getproj...,Project types: Get project types,11277
126,endpoint-get-orgs-org-id-project-types-id-getp...,endpoint-get-orgs-org-id-project-types-id-getp...,Project types: Get a project type,11088
284,endpoint-post-orgs-org-id-extract-createextrac...,endpoint-post-orgs-org-id-extract-createextract,Extract: Create extraction request,9722
...,...,...,...,...
52,endpoint-delete-orgs-org-id-strategy-factors-i...,endpoint-delete-orgs-org-id-strategy-factors-i...,Strategy: Delete a scoring factor,272
54,endpoint-delete-orgs-org-id-strategy-risks-id-...,endpoint-delete-orgs-org-id-strategy-risks-id-...,Strategy: Delete a risk,266
7,endpoint-delete-orgs-org-id-compliance-require...,endpoint-delete-orgs-org-id-compliance-require...,Compliance Maps: Delete a compliance requirement,220
6,endpoint-delete-orgs-org-id-compliance-regulat...,endpoint-delete-orgs-org-id-compliance-regulat...,Compliance Maps: Delete a compliance regulation,218


# 2. Preparando textos para embedding

In [27]:
def preparar_texto_embedding(documento):
    tags = ", ".join(documento["tags"])
    return f"HighBond API documentation\nTitle: {documento['title']}\nTags: {tags}\n\n{documento['conteudo']}"

textos_embedding = [preparar_texto_embedding(documento) for documento in documentos]

print(f"Documentos preparados: {len(textos_embedding)}")
print(textos_embedding[0][:500])

Documentos preparados: 369
HighBond API documentation
Title: Actions: Delete an action
Tags: endpoint, actions

## Actions

An action is a specific follow-up measure that is associated with an identified issue. You can add actions and assign action owners. You can also set up reminders for
yourself to retest issues or track hours spent on retesting by self-assigning actions.

> **Note —**
Only actions in projects that have an `"active"` state can be accessed using the API.

Learn more about actions.

### `DELETE /orgs/{or


# 3. Gerando os embeddings

In [28]:
import subprocess
import time
import requests
import sys

def iniciar_ollama_se_necessario():
    """Verifica se o Ollama está rodando; se não, tenta iniciar."""
    try:
        # Tenta conectar para ver se já está rodando
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama já está rodando.")
            return True
    except requests.exceptions.ConnectionError:
        print("🔄 Ollama não está rodando. Iniciando...")
        
        try:
            # Tenta iniciar o servidor Ollama em background
            if sys.platform == "win32":
                # No Windows, inicia sem janela de terminal
                subprocess.Popen(
                    ["ollama", "serve"],
                    creationflags=subprocess.CREATE_NO_WINDOW,
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL
                )
            else:
                # Linux/Mac
                subprocess.Popen(
                    ["ollama", "serve"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL
                )
            
            # Aguarda o servidor subir (até 30 segundos)
            for i in range(30):
                try:
                    response = requests.get("http://localhost:11434/api/tags", timeout=2)
                    if response.status_code == 200:
                        print(f"✅ Ollama iniciado com sucesso (tentativa {i+1})")
                        return True
                except:
                    time.sleep(1)
            
            print("❌ Timeout: Ollama não respondeu após 30 segundos")
            return False
            
        except FileNotFoundError:
            print("❌ Erro: 'ollama' não encontrado no PATH.")
            print("   Instale o Ollama: https://ollama.com/download")
            return False
        except Exception as e:
            print(f"❌ Erro ao iniciar Ollama: {e}")
            return False

# Executa a verificação/início
iniciar_ollama_se_necessario()


URL_OLLAMA = "http://localhost:11434/api/embed"
MODELO_EMBEDDING = "qwen3-embedding:0.6b"

✅ Ollama já está rodando.


In [29]:
# def gerar_embeddings(textos, tamanho_lote=4):
#     embeddings = []

#     for inicio in range(0, len(textos), tamanho_lote):
#         lote = textos[inicio:inicio + tamanho_lote]
#         response = requests.post(
#             URL_OLLAMA,
#             json={"model": MODELO_EMBEDDING, "input": lote},
#             timeout=1200
#         )
#         response.raise_for_status()
#         embeddings.extend(response.json()["embeddings"])
#         print(f"Processados: {min(inicio + tamanho_lote, len(textos))}/{len(textos)}")

#     matriz = np.asarray(embeddings, dtype=np.float32)

#     # Normaliza os vetores para utilizar produto escalar como similaridade do cosseno.
#     matriz /= np.linalg.norm(matriz, axis=1, keepdims=True)

#     return matriz

# matriz_embeddings = gerar_embeddings(textos_embedding)

# print(f"Formato da matriz: {matriz_embeddings.shape}")

In [30]:
# from pathlib import Path
# import json

# PASTA_INDICE = Path("indice")
# PASTA_INDICE.mkdir(exist_ok=True)

# np.save(PASTA_INDICE / "embeddings_qwen3_06b.npy", matriz_embeddings)

# with open(PASTA_INDICE / "documentos_ids.json", "w", encoding="utf-8") as arquivo:
#     json.dump([documento["id"] for documento in documentos], arquivo, ensure_ascii=False, indent=2)

In [31]:
# Carregando os embeddings salvos (célula autossuficiente: pode rodar após reiniciar o kernel)
from pathlib import Path
import numpy as np

CAMINHO_EMBEDDINGS = Path("indice") / "embeddings_qwen3_06b.npy"

if not CAMINHO_EMBEDDINGS.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {CAMINHO_EMBEDDINGS.resolve()}. "
        "Execute antes as células que geram e salvam os embeddings."
    )

matriz_embeddings = np.load(CAMINHO_EMBEDDINGS)

print(f"Matriz carregada: {matriz_embeddings.shape}")

Matriz carregada: (369, 1024)


# 5. Testando busca semântica

In [32]:
INSTRUCAO_BUSCA = (
    "Retrieve the most relevant HighBond API documentation for the question. "
    "The question may be in Portuguese or English and the documentation is in English."
)

def gerar_embedding_pergunta(pergunta):
    texto = f"Instruct: {INSTRUCAO_BUSCA}\nQuery: {pergunta}"

    response = requests.post(
        URL_OLLAMA,
        json={"model": MODELO_EMBEDDING, "input": texto},
        timeout=1200
    )
    response.raise_for_status()

    vetor = np.asarray(response.json()["embeddings"][0], dtype=np.float32)
    return vetor / np.linalg.norm(vetor)

In [33]:
def buscar_chunks(pergunta, quantidade=5, similaridade_minima=0.6):
    vetor_pergunta = gerar_embedding_pergunta(pergunta)
    similaridades = matriz_embeddings @ vetor_pergunta

    # Ordena, filtra pela similaridade mínima e mantém apenas a quantidade solicitada.
    indices = np.argsort(similaridades)[::-1]
    indices = indices[similaridades[indices] >= similaridade_minima][:quantidade]

    colunas = ["posição", "similaridade", "id", "title", "trecho"]

    resultados = [
        {
            "posição": posicao,
            "similaridade": round(float(similaridades[indice]), 4),
            "id": documentos[indice]["id"],
            "title": documentos[indice]["title"],
            "trecho": documentos[indice]["conteudo"].replace("\n", " ")
        }
        for posicao, indice in enumerate(indices, start=1)
    ]

    return pd.DataFrame(resultados, columns=colunas)

In [34]:
resultado = buscar_chunks(
    "Como listar os usuários pertencentes a uma organização?",
    quantidade=10,
    similaridade_minima=0.6
)

resultado

,posição,similaridade,id,title,trecho
0,1,0.8678,endpoint-get-orgs-org-id-strategy-users-getstr...,Strategy: Get users,## Strategy You can use the Strategy app to i...
1,2,0.8646,endpoint-get-orgs-org-id-users-getusers,Users: List users in an instance,## Users A user represents a member of HighBo...
2,3,0.8491,endpoint-get-orgs-org-id-users-uid-getuser,Users: Get user information,## Users A user represents a member of HighBo...
3,4,0.8305,endpoint-delete-orgs-org-id-users-uid-deleteuser,Users: Remove user from organization,## Users A user represents a member of HighBo...
4,5,0.7999,endpoint-get-orgs-org-id-groups-group-id-getgroup,Groups: Get group information,## Groups Groups are a security feature in La...
5,6,0.7985,endpoint-post-orgs-org-id-users-createuser,Users: Create a new user,## Users A user represents a member of HighBo...
6,7,0.7945,endpoint-get-orgs-org-id-groups-getgroups,Groups: List groups in an instance,## Groups Groups are a security feature in La...
7,8,0.7875,endpoint-get-orgs-org-id-strategy-users-id-get...,Strategy: Get a user,## Strategy You can use the Strategy app to i...
8,9,0.7705,endpoint-get-orgs-org-id-entities-getentities,Entities: Get entities,"## Entities Entities may be business units, d..."
9,10,0.7697,endpoint-post-orgs-org-id-groups-creategroup,Groups: Create a new group,## Groups Groups are a security feature in La...


In [35]:
buscar_chunks("crie um código que coleta todas as ações problemas e projetos",
    quantidade=10,
    similaridade_minima=0.6)

,posição,similaridade,id,title,trecho
0,1,0.7072,endpoint-get-orgs-org-id-projects-project-id-i...,Issues: Get issues,"## Issues An issue is a problem, control gap,..."
1,2,0.7069,endpoint-get-orgs-org-id-issues-getorgissues,Issues: Get an organization's issues,"## Issues An issue is a problem, control gap,..."
2,3,0.7011,endpoint-get-orgs-org-id-actions-id-getaction,Actions: Get an action,## Actions An action is a specific follow-up ...
3,4,0.6909,endpoint-delete-orgs-org-id-issues-id-deleteissue,Issues: Delete an issue,"## Issues An issue is a problem, control gap,..."
4,5,0.6855,endpoint-get-orgs-org-id-issues-id-getissue,Issues: Get an issue,"## Issues An issue is a problem, control gap,..."
5,6,0.6790,endpoint-post-orgs-org-id-projects-project-id-...,Issues: Create an issue,"## Issues An issue is a problem, control gap,..."
6,7,0.6788,endpoint-get-orgs-org-id-projects-getprojects,Projects: Get projects,## Projects A project is a management system ...
7,8,0.6737,endpoint-get-orgs-org-id-issues-issue-id-actio...,Actions: List actions in an issue,## Actions An action is a specific follow-up ...
8,9,0.6734,endpoint-get-orgs-org-id-projects-id-getproject,Projects: Get a project,## Projects A project is a management system ...
9,10,0.6713,endpoint-get-orgs-org-id-actions-action-id-com...,Actions: List action comments in an action,## Actions An action is a specific follow-up ...


In [44]:
buscar_chunks("paginação",
    quantidade=20,
    similaridade_minima=0.6)

,posição,similaridade,id,title,trecho
0,1,0.6616,endpoint-get-orgs-org-id-strategy-users-id-get...,Strategy: Get a user,## Strategy You can use the Strategy app to i...
1,2,0.6507,endpoint-get-orgs-org-id-interpretations-id-ge...,Interpretations: Get an interpretation,## Interpretations An interpretation is a bun...
2,3,0.6431,endpoint-get-orgs-org-id-strategy-users-getstr...,Strategy: Get users,## Strategy You can use the Strategy app to i...
3,4,0.6430,endpoint-get-orgs-org-id-entities-getentities,Entities: Get entities,"## Entities Entities may be business units, d..."
4,5,0.6398,endpoint-get-orgs-org-id-users-uid-getuser,Users: Get user information,## Users A user represents a member of HighBo...
5,6,0.6368,endpoint-get-orgs-org-id-robots-getrobots,Robots: Get a list of robots,## Robots A robot is a tool that lets you aut...
6,7,0.6349,endpoint-patch-orgs-org-id-import-data-session...,Importer: Retry import,"## Importer Importer for HighBond, responsibl..."
7,8,0.6328,endpoint-get-orgs-org-id-groups-group-id-getgroup,Groups: Get group information,## Groups Groups are a security feature in La...
8,9,0.6284,endpoint-get-orgs-org-id-import-session-id-sta...,Importer: Import Status,"## Importer Importer for HighBond, responsibl..."
9,10,0.6278,endpoint-get-orgs-org-id-storyboards-id-getsto...,Storyboards: Get a storyboard,## Storyboards A storyboard is a communicatio...
